## Final dataset preparation
- Now, ones we have done cleaning and feature engineering, let's perform some final tasks :
1. Remove outliers
2. Encode categorical variables (if any)
3. standarize / normalize the data
4. Remove unnecessary columns for feeding into models

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [21]:
df = pd.read_csv('../data/03_engineered/satellites_engineered.csv')

In [22]:
df.head()

,OBJECT_NAME,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,...,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,ORBIT_PERIOD_SEC,SEMI_MAJOR_AXIS,ORBIT_HEIGHT,PERIGEE,APOGEE,ORBITAL_SPEED,AGE_SINCE_LAUNCH,SAT_TYPE
0,CALSPHERE 1,2025-11-29 02:50:25.337184,13.763395,0.002678,90.2215,67.1407,174.9399,258.9789,900,4390,...,8.060000e-06,0.0,6277.520976,7354.966693,983.966693,957.272299,996.661088,7.361703,61.912,C
1,CALSPHERE 2,2025-11-29 03:05:54.157920,13.528812,0.002053,90.2363,71.0754,88.8358,28.0961,902,82921,...,3.700000e-07,0.0,6386.370100,7439.743866,1068.743866,1046.470816,1077.016916,7.319639,61.912,E
2,TEMPSAT 1,2025-11-29 03:03:30.714048,13.335805,0.007145,89.9891,212.6810,90.4072,62.5157,1512,93338,...,5.500000e-07,0.0,6478.798798,7511.354469,1140.354469,1079.682837,1187.026101,7.284664,60.912,E
3,CALSPHERE 4A,2025-11-29 04:04:06.381120,13.362355,0.006838,89.9087,124.3593,308.8371,226.1500,1520,93599,...,1.470000e-06,0.0,6465.926220,7501.401745,1130.401745,1072.108660,1174.694830,7.289495,60.912,H
4,OPS 5712 (P/L 160),2025-11-29 04:49:46.469856,14.738530,0.000482,69.9174,167.1531,291.9090,68.1543,2826,3672,...,8.985000e-05,0.0,5862.185654,7026.865500,655.865500,645.476443,652.254557,7.531610,58.912,A


----
### 1. Removing outliers
##### Key question : Why even remove outliers if we are going to perform anomaly detection?


### 1️⃣ Garbage-in → Garbage-out

* ML models (Isolation Forest, KMeans, etc.) assume that most of your data is “normal” to learn patterns.
* If your dataset accidentally has **obvious errors** (e.g., height = 0 km, speed = 1e6 m/s, negative values), the model may learn wrong patterns or consider normal data as anomalous.

---

### 2️⃣ Helps define “normal”

* Anomaly detection models define anomalies relative to what is normal.
* If there are **obvious outliers**, your baseline “normal” distribution will be skewed, reducing detection accuracy for true anomalies like orbital maneuvers.

---

### 3️⃣ Prevents false positives

* Spotting obvious errors ensures you **don’t flag bad data as anomalies**.
* Real anomalies should reflect **interesting satellite behavior**, not just bad CSV entries or measurement glitches.

---

✅ **Summary:**

* Spotting anomalies early is about **data quality**, not about defeating your anomaly detection goal.
* Once the dataset is clean, your ML models can focus on **real, subtle anomalies** (like unusual delta in orbital height, speed, or maneuvers).




---

## ❗For prototyping purpose, we will not perform outlier removal step for now, later on we may perform that!

### 2. Encode categorical variables 

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12696 entries, 0 to 12695
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   OBJECT_NAME        12696 non-null  object 
 1   EPOCH              12696 non-null  object 
 2   MEAN_MOTION        12696 non-null  float64
 3   ECCENTRICITY       12696 non-null  float64
 4   INCLINATION        12696 non-null  float64
 5   RA_OF_ASC_NODE     12696 non-null  float64
 6   ARG_OF_PERICENTER  12696 non-null  float64
 7   MEAN_ANOMALY       12696 non-null  float64
 8   NORAD_CAT_ID       12696 non-null  int64  
 9   REV_AT_EPOCH       12696 non-null  int64  
 10  BSTAR              12696 non-null  float64
 11  MEAN_MOTION_DOT    12696 non-null  float64
 12  MEAN_MOTION_DDOT   12696 non-null  float64
 13  ORBIT_PERIOD_SEC   12696 non-null  float64
 14  SEMI_MAJOR_AXIS    12696 non-null  float64
 15  ORBIT_HEIGHT       12696 non-null  float64
 16  PERIGEE            126

- We only have one categorical feature, that's going to be feed into the models and that is 'satellite type'.
- So we will use OneHotEncoder to encode the feature

#### ❗Removing the columns not requiring scaling

In [24]:
df = df.drop(columns=['OBJECT_NAME', 'EPOCH', 'NORAD_CAT_ID'])

In [25]:
from sklearn.preprocessing import OneHotEncoder

In [26]:
# Fit and transform
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder.fit_transform(df[['SAT_TYPE']])

# Get actual category names
encoded_cols = encoder.get_feature_names_out(['SAT_TYPE'])

# Convert to DataFrame with proper column names
df_encoded = pd.concat([df.drop('SAT_TYPE', axis=1), pd.DataFrame(X_encoded, columns=encoded_cols)], axis=1)


In [27]:
df_encoded.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,13.763395,0.002678,90.2215,67.1407,174.9399,258.9789,4390,0.000815,8.060000e-06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.528812,0.002053,90.2363,71.0754,88.8358,28.0961,82921,0.000042,3.700000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,13.335805,0.007145,89.9891,212.6810,90.4072,62.5157,93338,0.000091,5.500000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,13.362355,0.006838,89.9087,124.3593,308.8371,226.1500,93599,0.000265,1.470000e-06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14.738530,0.000482,69.9174,167.1531,291.9090,68.1543,3672,0.001379,8.985000e-05,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
# We combined Main dataframe (excluding the SAT_TYPE) + (X encoded (values) + encoded_cols feature names) 
X_encoded

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(12696, 24))

In [29]:
encoded_cols

array(['SAT_TYPE_A', 'SAT_TYPE_B', 'SAT_TYPE_C', 'SAT_TYPE_D',
       'SAT_TYPE_E', 'SAT_TYPE_F', 'SAT_TYPE_G', 'SAT_TYPE_H',
       'SAT_TYPE_J', 'SAT_TYPE_K', 'SAT_TYPE_L', 'SAT_TYPE_M',
       'SAT_TYPE_N', 'SAT_TYPE_P', 'SAT_TYPE_Q', 'SAT_TYPE_R',
       'SAT_TYPE_S', 'SAT_TYPE_T', 'SAT_TYPE_U', 'SAT_TYPE_V',
       'SAT_TYPE_W', 'SAT_TYPE_X', 'SAT_TYPE_Y', 'SAT_TYPE_Z'],
      dtype=object)

In [30]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12696 entries, 0 to 12695
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MEAN_MOTION        12696 non-null  float64
 1   ECCENTRICITY       12696 non-null  float64
 2   INCLINATION        12696 non-null  float64
 3   RA_OF_ASC_NODE     12696 non-null  float64
 4   ARG_OF_PERICENTER  12696 non-null  float64
 5   MEAN_ANOMALY       12696 non-null  float64
 6   REV_AT_EPOCH       12696 non-null  int64  
 7   BSTAR              12696 non-null  float64
 8   MEAN_MOTION_DOT    12696 non-null  float64
 9   MEAN_MOTION_DDOT   12696 non-null  float64
 10  ORBIT_PERIOD_SEC   12696 non-null  float64
 11  SEMI_MAJOR_AXIS    12696 non-null  float64
 12  ORBIT_HEIGHT       12696 non-null  float64
 13  PERIGEE            12696 non-null  float64
 14  APOGEE             12696 non-null  float64
 15  ORBITAL_SPEED      12696 non-null  float64
 16  AGE_SINCE_LAUNCH   126

---
### 3. Feature Scaling 

- **Numeric Features:**  
  Features like `SEMI_MAJOR_AXIS`, `ORBIT_HEIGHT`, `ORBITAL_SPEED`, etc., have different units and ranges. Scaling them using **StandardScaler** standardizes the values (mean=0, std=1) so that all numeric features contribute equally to distance-based algorithms like **KMeans** and **Isolation Forest**.

- **Categorical Features (One-Hot Encoded):**  
  Features like `SAT_TYPE` are converted into binary columns (0/1). These **do not require scaling**, as their values are already normalized and scaling would distort the categorical meaning.

- **Key Idea:**  
  - Scale numeric continuous features.  
  - Keep one-hot categorical features as-is.  
  This ensures the model correctly interprets distances and patterns without bias from different units.


In [31]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer # lets you apply different preprocessing to different columns

In [32]:
scaler = StandardScaler()

encoded_cols = [col for col in df_encoded.columns if col.startswith('SAT_TYPE_')]
numeric_cols = df_encoded.drop(columns = encoded_cols).columns

ct = ColumnTransformer([
    ('scaler', StandardScaler(), numeric_cols), # scale numeric feature
    ('pass', 'passthrough', encoded_cols) # keep encoded columns as is
])

df_scaled = ct.fit_transform(df_encoded)

In [33]:
df_scaled

array([[-2.02365967,  0.72145629,  1.35966487, ...,  0.        ,
         0.        ,  0.        ],
       [-2.40260608,  0.51185975,  1.36037974, ...,  0.        ,
         0.        ,  0.        ],
       [-2.71438976,  2.220199  ,  1.34843961, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.13931433,  0.30018335,  1.72267459, ...,  0.        ,
         0.        ,  0.        ],
       [-0.13818   ,  0.31333346,  1.72271323, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.76837185, -0.05141416, -0.50430345, ...,  0.        ,
         0.        ,  0.        ]], shape=(12696, 41))

In [34]:
# convert to dataframe                        # numeric_col is a series, so convert to a list 
df_scaled = pd.DataFrame(df_scaled, columns = numeric_cols.tolist() + encoded_cols)

In [35]:
df_scaled.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,-2.023660,0.721456,1.359665,-0.980813,0.133186,0.641852,-0.682557,0.142851,0.010382,-0.027989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-2.402606,0.511860,1.360380,-0.944014,-0.785752,-1.815440,5.221362,0.003565,0.007018,-0.027989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-2.714390,2.220199,1.348440,0.380358,-0.768982,-1.449111,6.004506,0.012454,0.007097,-0.027989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-2.671502,2.117011,1.344556,-0.445674,1.562190,0.292453,6.024128,0.043789,0.007499,-0.027989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.448421,-0.015017,0.378947,-0.045443,1.381527,-1.389099,-0.736536,0.244492,0.046157,-0.027989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


- Save as model ready dataset

In [36]:
df_scaled.to_csv('../data/04_scaled/satellites_scaled.csv', index=False)

- Save the encoder also (for later use in isolation forest)

In [37]:
import joblib

joblib.dump(encoder, '../data/04_scaled/encoder_sat_type.joblib')

['../data/04_scaled/encoder_sat_type.joblib']

#### Now we are ready for Machine Learning models!